# Redes Neuronales III – Práctica 3: Métricas de evaluación (Semana 7)

**Caso de uso (sector salud):** clasificación de tumores de mama como benignos o malignos a partir de 30 características de núcleos celulares (dataset *Breast Cancer Wisconsin – Diagnostic*, 569 muestras). Uso exclusivamente académico: no sustituye un diagnóstico médico.

Temas: ortogonalización, Accuracy, Precision, Recall, F1, Sensibilidad, Especificidad, matriz de confusión, ROC, ajuste de umbral.

## 1. Datos y distribución train/dev/test

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42

# 1. Carga: Breast Cancer Wisconsin (Diagnostic), incluido en scikit-learn
data = load_breast_cancer(as_frame=True)
X = data.data
y = (1 - data.target).astype("int32")      # 1 = maligno, 0 = benigno

# 2. Distribucion estratificada 60 / 20 / 20 (train / dev / test)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)
X_train, X_dev, y_train, y_dev = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=SEED)

# 3. Preprocesamiento: estandarizacion ajustada SOLO con train (evita fuga de datos)
scaler = StandardScaler().fit(X_train)
Xtr_s, Xdv_s, Xte_s = [scaler.transform(a).astype("float32")
                       for a in (X_train, X_dev, X_test)]
Xtr_raw = X_train.to_numpy(dtype="float32")     # sin escalar (solo para comparar)
Xdv_raw = X_dev.to_numpy(dtype="float32")
N_FEATURES = Xtr_s.shape[1]

resumen = pd.DataFrame({
    "conjunto": ["train", "dev", "test"],
    "muestras": [len(y_train), len(y_dev), len(y_test)],
    "malignos": [int(y_train.sum()), int(y_dev.sum()), int(y_test.sum())]})
resumen["% malignos"] = (100 * resumen["malignos"] / resumen["muestras"]).round(1)
print(resumen.to_string(index=False))
print("Caracteristicas:", N_FEATURES)

## 2. Modelo y utilidades

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

keras.utils.set_random_seed(SEED)

def build_model(units=(64, 32), use_bn=False, dropout=0.0, l1=0.0, l2=0.0, noise=0.0):
    """MLP binario. noise>0 agrega ruido gaussiano (data augmentation tabular)."""
    reg = regularizers.L1L2(l1=l1, l2=l2) if (l1 or l2) else None
    m = keras.Sequential([layers.Input(shape=(N_FEATURES,))])
    if noise:
        m.add(layers.GaussianNoise(noise))
    for u in units:
        m.add(layers.Dense(u, use_bias=not use_bn, kernel_regularizer=reg))
        if use_bn:
            m.add(layers.BatchNormalization())
        m.add(layers.Activation("relu"))
        if dropout:
            m.add(layers.Dropout(dropout))
    m.add(layers.Dense(1, activation="sigmoid"))
    return m

def compile_model(m, optimizer):
    m.compile(optimizer=optimizer, loss="binary_crossentropy",
              metrics=["accuracy",
                       keras.metrics.Recall(name="recall"),
                       keras.metrics.AUC(name="auc")])
    return m

def entrenar(make_model, make_opt, Xtr=None, Xdv=None, epochs=100, batch_size=32,
             callbacks=None, seeds=(1, 2, 3)):
    """Entrena con varias semillas y promedia, porque el dataset es pequeno."""
    Xtr = Xtr_s if Xtr is None else Xtr
    Xdv = Xdv_s if Xdv is None else Xdv
    filas, hist = [], None
    for s in seeds:
        keras.backend.clear_session()
        keras.utils.set_random_seed(s)
        m = compile_model(make_model(), make_opt())
        h = m.fit(Xtr, y_train, validation_data=(Xdv, y_dev), epochs=epochs,
                  batch_size=batch_size, callbacks=callbacks() if callbacks else [],
                  verbose=0)
        tr = m.evaluate(Xtr, y_train, verbose=0, return_dict=True)
        dv = m.evaluate(Xdv, y_dev, verbose=0, return_dict=True)
        filas.append([tr["accuracy"], dv["accuracy"], dv["recall"], dv["auc"],
                      len(h.history["loss"])])
        hist = h.history
    r = np.array(filas)
    resumen = {"acc_train": r[:, 0].mean(), "acc_dev": r[:, 1].mean(),
               "std_dev": r[:, 1].std(), "recall_dev": r[:, 2].mean(),
               "auc_dev": r[:, 3].mean(), "epocas": r[:, 4].mean()}
    return resumen, hist

## 3. Modelo final y métricas

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay, roc_curve,
                             roc_auc_score, classification_report)

# Modelo final: L2 + Dropout + Early Stopping (entrenado con train, validado con dev)
keras.backend.clear_session()
keras.utils.set_random_seed(SEED)
final = compile_model(build_model(units=(64, 32), dropout=0.3, l2=1e-3),
                      keras.optimizers.Adam(1e-3))
es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=15,
                                   restore_best_weights=True)
hist = final.fit(Xtr_s, y_train, validation_data=(Xdv_s, y_dev), epochs=200,
                 batch_size=32, callbacks=[es], verbose=0)
print("Epocas entrenadas:", len(hist.history["loss"]))

p_dev = final.predict(Xdv_s, verbose=0).ravel()    # probabilidad de malignidad
p_test = final.predict(Xte_s, verbose=0).ravel()

def metricas(y_true, p, umbral=0.5):
    y_pred = (p >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return {"accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall (sensibilidad)": recall_score(y_true, y_pred),
            "especificidad": tn / (tn + fp),
            "f1": f1_score(y_true, y_pred),
            "auc": roc_auc_score(y_true, p),
            "TN": tn, "FP": fp, "FN": fn, "TP": tp}

m_dev, m_test = metricas(y_dev, p_dev), metricas(y_test, p_test)
print(pd.DataFrame({"dev": m_dev, "test": m_test}).round(4))

## 4. Matriz de confusión y curva ROC (test)

In [ ]:
# Matriz de confusion y reporte por clase (test, umbral 0.5)
y_pred = (p_test >= 0.5).astype(int)
print(classification_report(y_test, y_pred, target_names=["Benigno", "Maligno"], digits=4))

ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=["Benigno", "Maligno"]).plot(cmap="Blues", values_format="d")
plt.title("Matriz de confusion - test (umbral 0.5)"); plt.show()

# Curva ROC
fpr, tpr, _ = roc_curve(y_test, p_test)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc_score(y_test, p_test):.3f}")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("1 - Especificidad (FPR)"); plt.ylabel("Sensibilidad (TPR)")
plt.title("Curva ROC - test"); plt.legend(); plt.grid(alpha=.3); plt.show()

## 5. Ajuste de umbral orientado a sensibilidad

In [ ]:
# Barrido de umbrales (la salida del modelo es una probabilidad de malignidad)
filas = []
for u in (0.01, 0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9):
    for nombre, yt, pp in (("dev", y_dev, p_dev), ("test", y_test, p_test)):
        tn, fp, fn, tp = confusion_matrix(yt, (pp >= u).astype(int)).ravel()
        filas.append({"umbral": u, "conjunto": nombre, "sensibilidad": tp / (tp + fn),
                      "especificidad": tn / (tn + fp), "FN": fn, "FP": fp})
print(pd.DataFrame(filas).round(3).to_string(index=False))

# Ajuste del umbral con DEV: en tamizaje interesa no perder casos malignos (FN costoso).
# Objetivo: sensibilidad >= 98 % en dev. Se toma el punto medio del rango de umbrales que
# lo cumplen (no el borde), porque con ~114 muestras en dev el borde es muy inestable.
umbrales = np.linspace(0.01, 0.99, 99)
admisibles = [u for u in umbrales if recall_score(y_dev, p_dev >= u) >= 0.98]
umbral = float(np.median(admisibles)) if admisibles else 0.5
print("Rango admisible en dev:", (round(min(admisibles), 2), round(max(admisibles), 2)) if admisibles else "ninguno")
print("Umbral elegido:", round(umbral, 3))

m_test_u = metricas(y_test, p_test, umbral)
comparacion = pd.DataFrame({"test (umbral 0.5)": m_test,
                            f"test (umbral {umbral:.3f})": m_test_u})
print(comparacion.round(4))

## 6. Análisis

*Cuántos falsos negativos hubo, qué implican en un contexto clínico y qué cambió al ajustar el umbral.*